# Indian/Maharashtra Skin Disease Model Training

This notebook trains a **bounded educational prototype**, not a medical diagnostic model. Use real, consented, legally usable images with reliable disease labels. The current ISIC 2016 folders containing lesion masks are not enough for named disease classification.

Required structure:
```text
dataset/
  train/Acne Vulgaris/*.jpg
  train/Eczema/*.jpg
  train/Fungal Infection/*.jpg
  train/Psoriasis/*.jpg
  train/Scabies/*.jpg
  train/Vitiligo/*.jpg
  train/Impetigo/*.jpg
  train/Warts/*.jpg
  train/Normal Skin/*.jpg
  train/Other Non Skin/*.jpg
  val/<same class folders>/...
```

Do not mix patient images between train and validation. For Maharashtra-specific performance, the validation set must contain separate Indian/Maharashtra phone-camera images.

In [ ]:
# Run this cell first. Colab already includes TensorFlow, but this keeps the runtime consistent.
!pip -q install scikit-learn seaborn

import json, os, shutil, zipfile
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
print('TensorFlow:', tf.__version__)

## 1. Add the dataset

Recommended for a large dataset: upload a ZIP to Google Drive and set `DATA_ROOT` below. For a small experiment, set `UPLOAD_ZIP = True` and upload a ZIP directly.

In [ ]:
UPLOAD_ZIP = True
DATA_ROOT = Path('/content/dataset')

if UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('Upload one ZIP containing train/ and val/ folders.')
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as z:
        z.extractall('/content/unpacked_dataset')
    candidates = [p for p in Path('/content/unpacked_dataset').rglob('train') if (p.parent / 'val').exists()]
    if not candidates:
        raise FileNotFoundError('ZIP must contain matching train/ and val/ folders.')
    DATA_ROOT = candidates[0].parent

TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
print('Dataset root:', DATA_ROOT)
print('Train:', TRAIN_DIR)
print('Validation:', VAL_DIR)

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
if not TRAIN_DIR.exists() or not VAL_DIR.exists():
    raise FileNotFoundError('Dataset must contain train/ and val/ directories.')
class_names = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
if len(class_names) < 2:
    raise ValueError('At least two disease/normal/non-skin class folders are required.')

def count_images(folder):
    return sum(1 for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

train_counts = {name: count_images(TRAIN_DIR / name) for name in class_names}
val_counts = {name: count_images(VAL_DIR / name) for name in class_names if (VAL_DIR / name).exists()}
missing_val = sorted(set(class_names) - set(val_counts))
if missing_val:
    raise ValueError(f'Missing validation folders: {missing_val}')
print('Classes:', class_names)
print('Train counts:', train_counts)
print('Validation counts:', val_counts)
if any(v < 20 for v in train_counts.values()):
    print('WARNING: some classes have fewer than 20 training images; results will be unstable.')

## 2. Load images and calculate class weights

Class weights reduce the tendency to predict the largest class repeatedly. They do not replace balanced, correctly labeled data.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int', shuffle=True, seed=SEED, class_names=class_names)
val_ds = tf.keras.utils.image_dataset_from_directory(VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int', shuffle=False, class_names=class_names)

y_train = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
weights = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(weights)}
print('Class weights:', class_weight)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# Raw 0..255 input is intentional: the Rescaling layer is embedded in the exported model.
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), dtype=tf.float32, name='image')
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.08)(x)
x = tf.keras.layers.RandomZoom(0.10)(x)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name='input_rescaling')(x)
base = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax', name='class_probabilities')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=callbacks)

# Fine-tune only the last MobileNetV2 layers at a small learning rate.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
fine_history = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight, callbacks=callbacks)

## 3. Evaluate before exporting

Do not deploy based only on accuracy. Check per-class recall, precision, macro-F1, and the confusion matrix.

In [ ]:
y_true, y_pred = [], []
for batch_x, batch_y in val_ds:
    probs = model.predict(batch_x, verbose=0)
    y_true.extend(batch_y.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.xticks(rotation=45, ha='right'); plt.show()

In [ ]:
# Export files compatible with this project.
OUT = Path('/content/exported_model')
OUT.mkdir(exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
(OUT / 'skin_model.tflite').write_bytes(tflite_bytes)
(OUT / 'labels.txt').write_text('\n'.join(class_names), encoding='utf-8')
metadata = {
    'source': 'Google Colab project-specific training',
    'architecture': 'MobileNetV2 transfer learning',
    'input_size': [224, 224],
    'input_preprocessing': 'raw_0_255',
    'embedded_preprocessing': 'Rescaling(1/127.5, offset=-1)',
    'classes': len(class_names),
    'class_names': class_names,
    'scope': 'Educational prototype; validate clinically before use'
}
(OUT / 'skin_model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Exported:', list(OUT.iterdir()))

from google.colab import files
for name in ['skin_model.tflite', 'labels.txt', 'skin_model_metadata.json']:
    files.download(str(OUT / name))